In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_ITO_Delhi_CPCB_2023.xlsx")

In [4]:
df


,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,225.0,164.0,152.0,76.0,71.0,NaN,60.0,81.0,114.0,130.0,351.0,376.0
1,2,405.0,166.0,215.0,123.0,67.0,NaN,56.0,77.0,113.0,128.0,345.0,352.0
2,3,415.0,200.0,125.0,144.0,77.0,NaN,100.0,80.0,112.0,124.0,440.0,313.0
3,4,390.0,278.0,103.0,72.0,79.0,136.0,155.0,189.0,113.0,139.0,387.0,293.0
4,5,400.0,269.0,113.0,68.0,136.0,110.0,87.0,86.0,181.0,145.0,405.0,275.0
5,6,439.0,317.0,129.0,83.0,191.0,101.0,75.0,78.0,102.0,201.0,372.0,283.0
6,7,414.0,431.0,157.0,81.0,79.0,196.0,61.0,79.0,171.0,188.0,355.0,332.0
7,8,422.0,167.0,264.0,93.0,102.0,135.0,51.0,94.0,83.0,133.0,461.0,341.0
8,9,486.0,NaN,112.0,181.0,125.0,117.0,66.0,101.0,49.0,160.0,455.0,321.0
9,10,466.0,372.0,186.0,123.0,193.0,117.0,53.0,117.0,38.0,146.0,309.0,329.0


In [5]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      27 non-null     float64
 4   April      33 non-null     float64
 5   May        22 non-null     float64
 6   June       32 non-null     float64
 7   July       34 non-null     float64
 8   August     34 non-null     float64
 9   September  34 non-null     float64
 10  October    36 non-null     float64
 11  November   33 non-null     float64
 12  December   32 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [8]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [9]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,225.0,164.0,152.000000,76.0,71.0,93.09375,60.000000,81.000000,114.000000,130.0,351.0,376.0
1,2,405.0,166.0,129.666667,123.0,67.0,93.09375,56.000000,77.000000,113.000000,128.0,345.0,352.0
2,3,415.0,200.0,125.000000,144.0,77.0,93.09375,100.000000,80.000000,112.000000,124.0,440.0,313.0
3,4,390.0,278.0,103.000000,72.0,79.0,136.00000,73.352941,101.352941,113.000000,139.0,387.0,293.0
4,5,400.0,269.0,113.000000,68.0,136.0,110.00000,87.000000,86.000000,100.735294,145.0,405.0,275.0


In [11]:
df_ml_ready.info()

df_ml_ready.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB


(31, 13)